# Schema linker Data

In [ ]:
import pandas as pd
import json
import ast
# Read the schema linking CSV file and filter for question_id 1344
schema_linking_df = pd.read_csv(
    '../../../../data/augmentation/decrease_naturalness/schema_linking/new_sql_nl_schema_queries_with_schema_linking_evaluated.csv')

schema_linking_df.columns


In [ ]:
# Questions with LL slr=1 but OL or LO slr=0
print("\n" + "="*80)
print("ADDITIONAL FILTER: LL has SLR=1 but AT LEAST ONE OL/LO has SLR=0")
print("="*80)
print("Finding questions where LL achieved perfect schema linking (slr=1)")
print("but AT LEAST ONE of the OL or LO columns has imperfect schema linking (slr=0)")
print("(Checking all three approaches: Full-Schema, TCSL, SCSL)\n")

# Function to check if a result string contains slr=0
def has_slr_0(result_str):
    try:
        if pd.isna(result_str):
            return False
        if isinstance(result_str, str):
            result_dict = ast.literal_eval(result_str)
        else:
            result_dict = result_str
        return result_dict.get('slr', 1) == 0
    except:
        return False

# Function to check if a result string contains slr=1
def has_slr_1(result_str):
    try:
        if pd.isna(result_str):
            return False
        if isinstance(result_str, str):
            result_dict = ast.literal_eval(result_str)
        else:
            result_dict = result_str
        return result_dict.get('slr', 0) == 1
    except:
        return False

# Work with all rows in schema_linking_df
slr1_questions = schema_linking_df.copy()

# Check LL columns for slr=1 (all three approaches: Full-Schema, TCSL, SCSL)
slr1_questions['LL_Full-Schema_slr1'] = slr1_questions['LL_Full-Schema_results'].apply(has_slr_1)
slr1_questions['LL_TCSL_slr1'] = slr1_questions['LL_TCSL_results'].apply(has_slr_1)
slr1_questions['LL_SCSL_slr1'] = slr1_questions['LL_SCSL_results'].apply(has_slr_1)

# Check OL/LO columns for slr=0 (all three approaches: Full-Schema, TCSL, SCSL)
slr1_questions['OL_Full-Schema_slr0'] = slr1_questions['OL_Full-Schema_results'].apply(has_slr_0)
slr1_questions['OL_TCSL_slr0'] = slr1_questions['OL_TCSL_results'].apply(has_slr_0)
slr1_questions['OL_SCSL_slr0'] = slr1_questions['OL_SCSL_results'].apply(has_slr_0)
slr1_questions['LO_Full-Schema_slr0'] = slr1_questions['LO_Full-Schema_results'].apply(has_slr_0)
slr1_questions['LO_TCSL_slr0'] = slr1_questions['LO_TCSL_results'].apply(has_slr_0)
slr1_questions['LO_SCSL_slr0'] = slr1_questions['LO_SCSL_results'].apply(has_slr_0)

# Create flags for each pair: LL has slr=1 AND corresponding OL/LO has slr=0
slr1_questions['OL_Full-Schema_improvement'] = slr1_questions['LL_Full-Schema_slr1'] & slr1_questions['OL_Full-Schema_slr0']
slr1_questions['OL_TCSL_improvement'] = slr1_questions['LL_TCSL_slr1'] & slr1_questions['OL_TCSL_slr0']
slr1_questions['OL_SCSL_improvement'] = slr1_questions['LL_SCSL_slr1'] & slr1_questions['OL_SCSL_slr0']
slr1_questions['LO_Full-Schema_improvement'] = slr1_questions['LL_Full-Schema_slr1'] & slr1_questions['LO_Full-Schema_slr0']
slr1_questions['LO_TCSL_improvement'] = slr1_questions['LL_TCSL_slr1'] & slr1_questions['LO_TCSL_slr0']
slr1_questions['LO_SCSL_improvement'] = slr1_questions['LL_SCSL_slr1'] & slr1_questions['LO_SCSL_slr0']

# Create method-level improvement flags:
# Full-Schema improvement: LL_Full-Schema=1 AND (OL_Full-Schema=0 OR LO_Full-Schema=0)
slr1_questions['Full-Schema_improvement'] = slr1_questions['LL_Full-Schema_slr1'] & (
    slr1_questions['OL_Full-Schema_slr0'] | slr1_questions['LO_Full-Schema_slr0']
)

# TCSL improvement: LL_TCSL=1 AND (OL_TCSL=0 OR LO_TCSL=0)
slr1_questions['TCSL_improvement'] = slr1_questions['LL_TCSL_slr1'] & (
    slr1_questions['OL_TCSL_slr0'] | slr1_questions['LO_TCSL_slr0']
)

# SCSL improvement: LL_SCSL=1 AND (OL_SCSL=0 OR LO_SCSL=0)
slr1_questions['SCSL_improvement'] = slr1_questions['LL_SCSL_slr1'] & (
    slr1_questions['OL_SCSL_slr0'] | slr1_questions['LO_SCSL_slr0']
)

# Combined flag: at least one method shows improvement (Full-Schema OR TCSL OR SCSL)
slr1_questions['has_improvement'] = (
    slr1_questions['Full-Schema_improvement'] |
    slr1_questions['TCSL_improvement'] |
    slr1_questions['SCSL_improvement']
)

# Filter for questions with improvement
improvement_questions = slr1_questions[slr1_questions['has_improvement']].copy()

if len(improvement_questions) > 0:
    print(f"Total questions with LL slr=1 AND corresponding OL/LO slr=0: {len(improvement_questions)}")

    improvement_qids = sorted(improvement_questions['question_id'].unique().tolist())
    print(f"\nQuestion IDs showing schema linking improvement: {improvement_qids}")

    print("\n" + "-"*80)
    print("BREAKDOWN BY SCHEMA LINKING METHOD:")
    print("-"*80)

    full_schema_improve = improvement_questions[improvement_questions['Full-Schema_improvement']]['question_id'].tolist()
    tcsl_improve = improvement_questions[improvement_questions['TCSL_improvement']]['question_id'].tolist()
    scsl_improve = improvement_questions[improvement_questions['SCSL_improvement']]['question_id'].tolist()

    print(f"\nFull-Schema improvement (LL_Full-Schema=1 & (OL_Full-Schema=0 OR LO_Full-Schema=0)): {len(full_schema_improve)} questions")
    print(f"  Question IDs: {sorted(full_schema_improve)}")

    print(f"\nTCSL improvement (LL_TCSL=1 & (OL_TCSL=0 OR LO_TCSL=0)): {len(tcsl_improve)} questions")
    print(f"  Question IDs: {sorted(tcsl_improve)}")

    print(f"\nSCSL improvement (LL_SCSL=1 & (OL_SCSL=0 OR LO_SCSL=0)): {len(scsl_improve)} questions")
    print(f"  Question IDs: {sorted(scsl_improve)}")

    print("\n" + "-"*80)
    print("DETAILED BREAKDOWN BY SPECIFIC COLUMN PAIRS:")
    print("-"*80)

    ol_full_improve = improvement_questions[improvement_questions['OL_Full-Schema_improvement']]['question_id'].tolist()
    ol_tcsl_improve = improvement_questions[improvement_questions['OL_TCSL_improvement']]['question_id'].tolist()
    ol_scsl_improve = improvement_questions[improvement_questions['OL_SCSL_improvement']]['question_id'].tolist()
    lo_full_improve = improvement_questions[improvement_questions['LO_Full-Schema_improvement']]['question_id'].tolist()
    lo_tcsl_improve = improvement_questions[improvement_questions['LO_TCSL_improvement']]['question_id'].tolist()
    lo_scsl_improve = improvement_questions[improvement_questions['LO_SCSL_improvement']]['question_id'].tolist()

    print(f"\nOL_Full-Schema improvement (LL_Full-Schema=1 & OL_Full-Schema=0): {len(ol_full_improve)} questions")
    print(f"  Question IDs: {sorted(ol_full_improve)}")

    print(f"\nOL_TCSL improvement (LL_TCSL=1 & OL_TCSL=0): {len(ol_tcsl_improve)} questions")
    print(f"  Question IDs: {sorted(ol_tcsl_improve)}")

    print(f"\nOL_SCSL improvement (LL_SCSL=1 & OL_SCSL=0): {len(ol_scsl_improve)} questions")
    print(f"  Question IDs: {sorted(ol_scsl_improve)}")

    print(f"\nLO_Full-Schema improvement (LL_Full-Schema=1 & LO_Full-Schema=0): {len(lo_full_improve)} questions")
    print(f"  Question IDs: {sorted(lo_full_improve)}")

    print(f"\nLO_TCSL improvement (LL_TCSL=1 & LO_TCSL=0): {len(lo_tcsl_improve)} questions")
    print(f"  Question IDs: {sorted(lo_tcsl_improve)}")

    print(f"\nLO_SCSL improvement (LL_SCSL=1 & LO_SCSL=0): {len(lo_scsl_improve)} questions")
    print(f"  Question IDs: {sorted(lo_scsl_improve)}")

    # Detailed table
    print("\n" + "="*80)
    print("DETAILED TABLE - SCHEMA LINKING IMPROVEMENT:")
    print("="*80)
    print("(Showing method-level improvements and specific column details)\n")

    detail_cols = ['question_id', 'db_id',
                  'Full-Schema_improvement', 'TCSL_improvement', 'SCSL_improvement',
                  'OL_Full-Schema_improvement', 'LO_Full-Schema_improvement',
                  'OL_TCSL_improvement', 'LO_TCSL_improvement',
                  'OL_SCSL_improvement', 'LO_SCSL_improvement',
                  'LL_Full-Schema_slr1', 'LL_TCSL_slr1', 'LL_SCSL_slr1']
    detail_df = improvement_questions[detail_cols].copy()
    detail_df.columns = ['Q_ID', 'DB',
                        'Full_IMP', 'TCSL_IMP', 'SCSL_IMP',
                        'OL_Full', 'LO_Full',
                        'OL_TCSL', 'LO_TCSL',
                        'OL_SCSL', 'LO_SCSL',
                        'LL_Full', 'LL_TCSL', 'LL_SCSL']
    print(detail_df.to_string(index=False))

    # Summary statistics
    print("\n" + "-"*80)
    print("SUMMARY STATISTICS:")
    print("-"*80)

    # Questions with improvement in ALL THREE methods
    all_three_methods = improvement_questions[
        improvement_questions['Full-Schema_improvement'] &
        improvement_questions['TCSL_improvement'] &
        improvement_questions['SCSL_improvement']
    ]
    print(f"Questions with improvement in ALL THREE methods (Full-Schema & TCSL & SCSL): {len(all_three_methods)}")
    if len(all_three_methods) > 0:
        print(f"  Question IDs: {sorted(all_three_methods['question_id'].tolist())}")

    # Questions with improvement in BOTH TCSL AND SCSL
    both_tcsl_scsl = improvement_questions[
        improvement_questions['TCSL_improvement'] &
        improvement_questions['SCSL_improvement']
    ]
    print(f"\nQuestions with improvement in BOTH TCSL & SCSL: {len(both_tcsl_scsl)}")
    if len(both_tcsl_scsl) > 0:
        print(f"  Question IDs: {sorted(both_tcsl_scsl['question_id'].tolist())}")

    # Questions with Full-Schema improvement only
    full_only = improvement_questions[
        improvement_questions['Full-Schema_improvement'] &
        ~improvement_questions['TCSL_improvement'] &
        ~improvement_questions['SCSL_improvement']
    ]
    print(f"\nQuestions with Full-Schema improvement only: {len(full_only)}")
    if len(full_only) > 0:
        print(f"  Question IDs: {sorted(full_only['question_id'].tolist())}")

    # Questions with TCSL improvement only
    tcsl_only = improvement_questions[
        ~improvement_questions['Full-Schema_improvement'] &
        improvement_questions['TCSL_improvement'] &
        ~improvement_questions['SCSL_improvement']
    ]
    print(f"\nQuestions with TCSL improvement only: {len(tcsl_only)}")
    if len(tcsl_only) > 0:
        print(f"  Question IDs: {sorted(tcsl_only['question_id'].tolist())}")

    # Questions with SCSL improvement only
    scsl_only = improvement_questions[
        ~improvement_questions['Full-Schema_improvement'] &
        ~improvement_questions['TCSL_improvement'] &
        improvement_questions['SCSL_improvement']
    ]
    print(f"\nQuestions with SCSL improvement only: {len(scsl_only)}")
    if len(scsl_only) > 0:
        print(f"  Question IDs: {sorted(scsl_only['question_id'].tolist())}")

    # Questions with improvement in ALL OL columns (across all three methods)
    all_ol_improve = improvement_questions[
        improvement_questions['OL_Full-Schema_improvement'] &
        improvement_questions['OL_TCSL_improvement'] &
        improvement_questions['OL_SCSL_improvement']
    ]
    print(f"\nQuestions with improvement in ALL OL columns (Full-Schema & TCSL & SCSL): {len(all_ol_improve)}")
    if len(all_ol_improve) > 0:
        print(f"  Question IDs: {sorted(all_ol_improve['question_id'].tolist())}")

    # Questions with improvement in ALL LO columns (across all three methods)
    all_lo_improve = improvement_questions[
        improvement_questions['LO_Full-Schema_improvement'] &
        improvement_questions['LO_TCSL_improvement'] &
        improvement_questions['LO_SCSL_improvement']
    ]
    print(f"\nQuestions with improvement in ALL LO columns (Full-Schema & TCSL & SCSL): {len(all_lo_improve)}")
    if len(all_lo_improve) > 0:
        print(f"  Question IDs: {sorted(all_lo_improve['question_id'].tolist())}")

    # Intersection Analysis
    print("\n" + "="*80)
    print("INTERSECTION ANALYSIS:")
    print("="*80)

    # Intersection: ALL THREE methods
    all_three_methods = improvement_questions[
        improvement_questions['Full-Schema_improvement'] &
        improvement_questions['TCSL_improvement'] &
        improvement_questions['SCSL_improvement']
    ]
    print(f"\nIntersection of Full-Schema ∩ TCSL ∩ SCSL improvements: {len(all_three_methods)}")
    if len(all_three_methods) > 0:
        print(f"  Question IDs: {sorted(all_three_methods['question_id'].tolist())}")

    # Intersection: TCSL improvement AND SCSL improvement
    tcsl_and_scsl = improvement_questions[
        improvement_questions['TCSL_improvement'] &
        improvement_questions['SCSL_improvement']
    ]
    print(f"\nIntersection of TCSL ∩ SCSL improvements: {len(tcsl_and_scsl)}")
    if len(tcsl_and_scsl) > 0:
        print(f"  Question IDs: {sorted(tcsl_and_scsl['question_id'].tolist())}")

    # Intersection: Full-Schema AND TCSL
    full_and_tcsl = improvement_questions[
        improvement_questions['Full-Schema_improvement'] &
        improvement_questions['TCSL_improvement']
    ]
    print(f"\nIntersection of Full-Schema ∩ TCSL improvements: {len(full_and_tcsl)}")
    if len(full_and_tcsl) > 0:
        print(f"  Question IDs: {sorted(full_and_tcsl['question_id'].tolist())}")

    # Intersection: Full-Schema AND SCSL
    full_and_scsl = improvement_questions[
        improvement_questions['Full-Schema_improvement'] &
        improvement_questions['SCSL_improvement']
    ]
    print(f"\nIntersection of Full-Schema ∩ SCSL improvements: {len(full_and_scsl)}")
    if len(full_and_scsl) > 0:
        print(f"  Question IDs: {sorted(full_and_scsl['question_id'].tolist())}")

    # Intersection: OL improvements (ALL THREE methods)
    ol_all_three = improvement_questions[
        improvement_questions['OL_Full-Schema_improvement'] &
        improvement_questions['OL_TCSL_improvement'] &
        improvement_questions['OL_SCSL_improvement']
    ]
    print(f"\nIntersection of OL_Full-Schema ∩ OL_TCSL ∩ OL_SCSL improvements: {len(ol_all_three)}")
    if len(ol_all_three) > 0:
        print(f"  Question IDs: {sorted(ol_all_three['question_id'].tolist())}")

    # Intersection: LO improvements (ALL THREE methods)
    lo_all_three = improvement_questions[
        improvement_questions['LO_Full-Schema_improvement'] &
        improvement_questions['LO_TCSL_improvement'] &
        improvement_questions['LO_SCSL_improvement']
    ]
    print(f"\nIntersection of LO_Full-Schema ∩ LO_TCSL ∩ LO_SCSL improvements: {len(lo_all_three)}")
    if len(lo_all_three) > 0:
        print(f"  Question IDs: {sorted(lo_all_three['question_id'].tolist())}")

    # Intersection: OL and LO for Full-Schema
    ol_and_lo_full = improvement_questions[
        improvement_questions['OL_Full-Schema_improvement'] &
        improvement_questions['LO_Full-Schema_improvement']
    ]
    print(f"\nIntersection of OL_Full-Schema ∩ LO_Full-Schema improvements: {len(ol_and_lo_full)}")
    if len(ol_and_lo_full) > 0:
        print(f"  Question IDs: {sorted(ol_and_lo_full['question_id'].tolist())}")

    # Intersection: OL and LO for TCSL
    ol_and_lo_tcsl = improvement_questions[
        improvement_questions['OL_TCSL_improvement'] &
        improvement_questions['LO_TCSL_improvement']
    ]
    print(f"\nIntersection of OL_TCSL ∩ LO_TCSL improvements: {len(ol_and_lo_tcsl)}")
    if len(ol_and_lo_tcsl) > 0:
        print(f"  Question IDs: {sorted(ol_and_lo_tcsl['question_id'].tolist())}")

    # Intersection: OL and LO for SCSL
    ol_and_lo_scsl = improvement_questions[
        improvement_questions['OL_SCSL_improvement'] &
        improvement_questions['LO_SCSL_improvement']
    ]
    print(f"\nIntersection of OL_SCSL ∩ LO_SCSL improvements: {len(ol_and_lo_scsl)}")
    if len(ol_and_lo_scsl) > 0:
        print(f"  Question IDs: {sorted(ol_and_lo_scsl['question_id'].tolist())}")

    # Intersection: ALL SIX column improvements (complete improvement)
    all_six = improvement_questions[
        improvement_questions['OL_Full-Schema_improvement'] &
        improvement_questions['OL_TCSL_improvement'] &
        improvement_questions['OL_SCSL_improvement'] &
        improvement_questions['LO_Full-Schema_improvement'] &
        improvement_questions['LO_TCSL_improvement'] &
        improvement_questions['LO_SCSL_improvement']
    ]
    print(f"\nIntersection of ALL 6 column improvements (complete improvement across all): {len(all_six)}")
    if len(all_six) > 0:
        print(f"  Question IDs: {sorted(all_six['question_id'].tolist())}")

    # Summary table showing different intersection patterns
    print("\n" + "-"*80)
    print("INTERSECTION SUMMARY TABLE:")
    print("-"*80)
    print(f"{'Category':<60} {'Count':>10}")
    print("-"*80)
    print(f"{'Full-Schema ∩ TCSL ∩ SCSL (all three methods)':<60} {len(all_three_methods):>10}")
    print(f"{'Full-Schema ∩ TCSL':<60} {len(full_and_tcsl):>10}")
    print(f"{'Full-Schema ∩ SCSL':<60} {len(full_and_scsl):>10}")
    print(f"{'TCSL ∩ SCSL':<60} {len(tcsl_and_scsl):>10}")
    print(f"{'OL_Full-Schema ∩ OL_TCSL ∩ OL_SCSL (all OL improvements)':<60} {len(ol_all_three):>10}")
    print(f"{'LO_Full-Schema ∩ LO_TCSL ∩ LO_SCSL (all LO improvements)':<60} {len(lo_all_three):>10}")
    print(f"{'OL_Full-Schema ∩ LO_Full-Schema (Full-Schema across OL/LO)':<60} {len(ol_and_lo_full):>10}")
    print(f"{'OL_TCSL ∩ LO_TCSL (TCSL across OL/LO)':<60} {len(ol_and_lo_tcsl):>10}")
    print(f"{'OL_SCSL ∩ LO_SCSL (SCSL across OL/LO)':<60} {len(ol_and_lo_scsl):>10}")
    print(f"{'ALL 6 columns (complete improvement)':<60} {len(all_six):>10}")
    print("-"*80)


else:
    print("\nNo questions found with this pattern")

print("="*80)


In [ ]:
# Display the filtered data for question_id X for comparison
qid = 245
filtered_df = schema_linking_df[schema_linking_df['question_id'] == qid]

print("="*80)
print(f"QUESTION ID {qid} - DETAILED COMPARISON")
print("="*80)

if not filtered_df.empty:
    row = filtered_df.iloc[0]

    print("\n" + "="*80)
    print("BASIC INFORMATION")
    print("="*80)
    print(f"Question ID: {row['question_id']}")
    print(f"Database ID: {row['db_id']}")

    print("\n" + "="*80)
    print("QUESTIONS")
    print("="*80)
    print(f"Original Question:\n  {row['original_question']}")
    print(f"\nNew Question:\n  {row['new_question']}")

    print("\n" + "="*80)
    print("EVIDENCE")
    print("="*80)
    print(f"Original Evidence:\n  {row['original_evidence']}")
    print(f"\nNew Evidence:\n  {row['new_evidence']}")

    print("\n" + "="*80)
    print("SQL QUERIES")
    print("="*80)
    print(f"Original SQL:\n  {row['original_sql']}")
    print(f"\nNew SQL:\n  {row['new_sql']}")

    print("\n" + "="*80)
    print("GOLD SCHEMAS")
    print("="*80)
    print(f"Original Gold Schema:\n  {row['original_gold_schema']}")
    print(f"\nNew Gold Schema:\n  {row['new_gold_schema']}")

    print("\n" + "="*80)
    print("EXECUTION ACCURACY (EX) SCORES")
    print("="*80)
    print("OL (Original NL → Original SQL):")
    print(f"  Full-Schema: {row['OL_Full-Schema']}")
    print(f"  TCSL: {row['OL_TCSL']}")
    print(f"  SCSL: {row['OL_SCSL']}")

    print("\nLO (Less Natural NL → Original SQL):")
    print(f"  Full-Schema: {row['LO_Full-Schema']}")
    print(f"  TCSL: {row['LO_TCSL']}")
    print(f"  SCSL: {row['LO_SCSL']}")

    print("\nLL (Less Natural NL → Less Natural SQL):")
    print(f"  Full-Schema: {row['LL_Full-Schema']}")
    print(f"  TCSL: {row['LL_TCSL']}")
    print(f"  SCSL: {row['LL_SCSL']}")

    print("\n" + "="*80)
    print("DETAILED RESULTS")
    print("="*80)
    print("OL Results:")
    print(f"  Full-Schema: {row['OL_Full-Schema_results']}")
    print(f"  TCSL: {row['OL_TCSL_results']}")
    print(f"  SCSL: {row['OL_SCSL_results']}")

    print("\nLO Results:")
    print(f"  Full-Schema: {row['LO_Full-Schema_results']}")
    print(f"  TCSL: {row['LO_TCSL_results']}")
    print(f"  SCSL: {row['LO_SCSL_results']}")

    print("\nLL Results:")
    print(f"  Full-Schema: {row['LL_Full-Schema_results']}")
    print(f"  TCSL: {row['LL_TCSL_results']}")
    print(f"  SCSL: {row['LL_SCSL_results']}")

    print("\n" + "="*80)
    print("SUMMARY ANALYSIS")
    print("="*80)

    # Check which scenarios apply
    ol_failed = row['OL_Full-Schema'] == 0
    lo_failed = row['LO_Full-Schema'] == 0
    ll_success = row['LL_Full-Schema'] == 1

    print(f"OL Failed (Original failed): {ol_failed}")
    print(f"LO Failed (Less natural NL failed): {lo_failed}")
    print(f"LL Success (Both less natural succeeded): {ll_success}")

    if ll_success and ol_failed and not lo_failed:
        print("\n→ SCENARIO 1: Benefits from naturalness decrease in NL only")
    elif ll_success and lo_failed and not ol_failed:
        print("\n→ SCENARIO 2: Benefits from naturalness decrease in SQL only")
    elif ll_success and ol_failed and lo_failed:
        print("\n→ SCENARIO 3: Requires BOTH NL and SQL naturalness decrease")
    else:
        print("\n→ Does not fit main scenarios")

    print("="*80)
else:
    print("No data found for question_id 1344")
